# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the protein mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities such as record sets, fields, and columns are referenced by their Croissant `@id` values to ensure clarity and reproducibility.

### Dataset Source
The dataset source is defined by the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
We load metadata and records from the dataset using `mlcroissant`. The Croissant schema captures both data and metadata:

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Croissant Dataset object

print(f"{metadata.name}: {metadata.description}")
print(f"Citation: {metadata.cite_as}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references are by Croissant `@id`.

`mlcroissant` exposes record sets via the `.record_sets` attribute. We will display their `@id` and field information.

In [ ]:
# List all record sets and their fields
for rs in dataset.record_sets:
    print(f"Record set name: {rs.name}  @id: {rs.id}")
    for field in rs.fields:
        print(f"  Field: {field.name}  @id: {field.id}  Data type: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**Choose record sets and fields using their Croissant `@id`.**

We will extract records for **each record set** and load them into Pandas DataFrames. First, we collect all record set `@id`s.

In [ ]:
# Get all record set @id values
record_sets_ids = [rs.id for rs in dataset.record_sets]
print("Available record set @id values:")
for rsid in record_sets_ids:
    print(" ", rsid)

# Load each record set into a dataframe
dataframes = {}
for rs in dataset.record_sets:
    recs = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(recs)

# Display columns for the first record set as example
if record_sets_ids:
    first_id = record_sets_ids[0]
    print(f"\nFields (columns) for record set {first_id}:")
    print(dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric columns, and grouping.

**You must use Croissant `@id` for referencing fields/columns in all operations.**

*We will select a representative numeric field (e.g., peptide_count, molecular_weight, or abundance, whichever is present), filter to high values, normalize, and group by a categorical attribute if available.*

In [ ]:
# Pick the main record set for protein data - customize as needed based on previous output
main_record_set_id = record_sets_ids[0] if record_sets_ids else None
# Inspect all fields for this record set
print(f"Fields in record set {main_record_set_id}:")
columns = dataframes[main_record_set_id].columns.tolist() if main_record_set_id else []
pprint(columns)

# Choose a relevant numeric field (by @id)
numeric_field_candidates = [col for col in columns if 'peptide' in col.lower() or 'abundance' in col.lower() or 'weight' in col.lower() or 'count' in col.lower() or 'coverage' in col.lower() or 'mw' in col.lower()]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
else:
    numeric_field = columns[0] if columns else None
print(f"Using numeric field: {numeric_field}")

# Filtering
if numeric_field:
    # Try to cast values to numeric, coerce errors
    df = dataframes[main_record_set_id].copy()
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > mean ({threshold}): {len(filtered_df)} records\n")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (first 5 rows):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if available
    group_candidates = [col for col in columns if col != numeric_field and df[col].nunique() < 10 and df[col].dtype == object]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())
    else:
        print("No categorical field with <10 unique values found for grouping.")
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and if possible, show group means.

*All axes must reference fields by Croissant `@id`.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field in filtered_df:
        plt.figure(figsize=(8, 4))
        order = filtered_df[group_field].value_counts().index
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field], order=order)
        plt.title(f"{numeric_field} grouped by {group_field} (@id)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated loading and exploring the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` dataset using Croissant schemas and the `mlcroissant` Python library.

- All references to record sets, fields, and groupings have used their canonical Croissant `@id` values.
- Filtering, normalization, and grouping operations were shown on a representative numeric field.
- Visualizations highlighted core distributions and category breakdowns (where relevant).

This workflow can be adapted for deeper protein feature engineering, biomarker discovery, and machine learning pipelines using the standardized, FAIR-structured data documentation in Croissant.